# M3L4 E09 — LangGraph router + Langfuse [OK] Resolution
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

## ¿Por qué importa este ejercicio?

En E08 usamos un grafo lineal (un solo nodo). Pero en producción los agentes tienen que **decidir qué hacer según la consulta**. Ahí entra el **routing condicional**.

Vamos a construir un grafo con:
- Un **router** que clasifica la consulta
- **Nodos especialistas** (HR, IT, Finance, Legal) que responden
- **Edges condicionales** que dirigen el flujo según el intent

Y Langfuse traceará automáticamente qué ruta tomó cada consulta.

| Concepto | Definición simple | Cómo aparece acá |
|---|---|---|
| **Conditional edge** | Arista que decide a qué nodo ir según el estado | `add_conditional_edges('router_node', route_to_node)` |
| **Router** | Nodo que clasifica la consulta | `router_node()` llama a `route_query_v2()` |
| **Agente especialista** | Nodo que responde según el dominio | `hr_node`, `it_node`, `finance_node`, `legal_node` |

In [ ]:
!pip install -q langfuse langchain langchain-openai langgraph
print('Instalación completa.')

In [ ]:
import os
from getpass import getpass

os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key: ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key: ')
os.environ['LANGFUSE_BASE_URL']   = 'https://cloud.langfuse.com'
os.environ['OPENAI_API_KEY']      = getpass('OpenAI API Key: ')
print('OK.')

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END
from langfuse.langchain import CallbackHandler

def route_query_v2(query):
    q = query.lower()
    hr_kw      = ['vacaciones','licencia','recibo','nómina','rrhh']
    it_kw      = ['vpn','error','app','laptop','wifi','login','contraseña']
    finance_kw = ['factura','pago','reembolso','gasto','cobro','comprobante','salario']
    legal_kw   = ['contrato','legal','confidencialidad','nda','acuerdo']
    det = []
    if any(w in q for w in hr_kw): det.append('hr')
    if any(w in q for w in it_kw): det.append('it')
    if any(w in q for w in finance_kw): det.append('finance')
    if any(w in q for w in legal_kw): det.append('legal')
    if len(det) > 1: return 'multi_intent'
    if len(det) == 1: return det[0]
    if len(q.split()) <= 2: return 'clarification'
    return 'general'

class AgentState(TypedDict):
    query: str
    intent: str
    response: str

print('Setup listo.')

In [ ]:
def router_node(state): return {'intent': route_query_v2(state['query'])}
def hr_node(state):      return {'response': 'HRAgent: Para vacaciones o recibos, ingresá al portal de RRHH.'}
def it_node(state):      return {'response': 'ITAgent: Para VPN o errores técnicos, revisá conexión y abrí ticket.'}
def finance_node(state): return {'response': 'FinanceAgent: Para facturas o pagos, revisá el portal de facturación.'}
def legal_node(state):   return {'response': 'LegalAgent: Para contratos o NDAs, contactá al equipo legal.'}
def general_node(state): return {'response': 'GeneralAgent: Necesito más información para ayudarte. ¿Podés ser más específico?'}

def route_to_node(state):
    return {
        'hr':           'hr_node',
        'it':           'it_node',
        'finance':      'finance_node',
        'legal':        'legal_node',
        'multi_intent': 'general_node',
        'clarification':'general_node',
    }.get(state['intent'], 'general_node')

print('Nodos listos.')

## Solución — Construcción del grafo con routing

El grafo tiene esta estructura:

```
START -> router_node -> conditional_edge -> {hr_node, it_node, finance_node, legal_node, general_node} -> END
```

La función `route_to_node()` actúa como el **edge condicional**: mira el `intent` en el estado y dirige a la rama correspondiente.

In [ ]:
builder = StateGraph(AgentState)
builder.add_node('router_node',  router_node)
builder.add_node('hr_node',      hr_node)
builder.add_node('it_node',      it_node)
builder.add_node('finance_node', finance_node)
builder.add_node('legal_node',   legal_node)
builder.add_node('general_node', general_node)
builder.set_entry_point('router_node')
builder.add_conditional_edges(
    'router_node',
    route_to_node,
    {'hr_node': 'hr_node', 'it_node': 'it_node', 'finance_node': 'finance_node',
     'legal_node': 'legal_node', 'general_node': 'general_node'}
)
for node in ['hr_node','it_node','finance_node','legal_node','general_node']:
    builder.add_edge(node, END)
graph = builder.compile()
print('Grafo compilado.')
print(graph.get_graph().draw_mermaid())

## Ejecución con Langfuse

Cada consulta genera un trace en Langfuse que muestra:
- El nodo `router_node` con input/output
- El nodo especialista al que derivó
- La respuesta final

In [ ]:
queries = [
    '¿Cómo solicito mis días de vacaciones?',
    'Mi VPN no conecta desde ayer',
    'Necesito ver mi factura del mes pasado',
    'Necesito el contrato de confidencialidad actualizado',
    'ayuda'
]

for i, q in enumerate(queries):
    lf = CallbackHandler()
    output = graph.invoke(
        {'query': q, 'intent': '', 'response': ''},
        config={
            'callbacks': [lf],
            'metadata': {'langfuse_tags': ['m3l4', 'router-demo'], 'langfuse_user_id': f'student-{i}'}
        }
    )
    print(f'Query: {q[:45]}')
    print(f'  Intent: {output["intent"]} | {output["response"][:60]}...')
    print()

## Verificación

In [ ]:
lf = CallbackHandler()
r = graph.invoke({'query': 'No puedo ver mi factura', 'intent': '', 'response': ''},
                 config={'callbacks': [lf]})
assert r['intent'] == 'finance'
assert len(r['response']) > 5
print('Checks E09 OK')

## [OK] Cierre — ¿Qué logramos?

| Componente | Proósito |
|---|---|
| **Router node** | Clasifica la consulta en un dominio |
| **Conditional edge** | Decide qué agente especialista ejecutar |
| **Nodos especialistas** | Responden según su dominio de conocimiento |
| **Langfuse** | Tracea automáticamente la ruta elegida |

**¿Qué sigue?** En E10 vamos a agregar un **supervisor** que controla el flujo entre agentes y previene loops.